# 02 Predictions And External Evaluation

Generated by `scripts/revision/create_revision_notebooks.py`.


## Setup

In [ ]:
from pathlib import Path
import json
import os
import subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Drive mount skipped or already mounted: {exc}')

DRIVE_ROOT = Path('/content/drive/MyDrive/ECG-Ramba')
REPO_DIR = DRIVE_ROOT / 'ECG-RAMBA'
if not (REPO_DIR / 'configs' / 'config.py').exists():
    REPO_DIR = Path.cwd()

os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.chdir(REPO_DIR)

print('cwd       :', Path.cwd())
print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)

def run(cmd, check=True):
    print(f'$ {cmd}')
    return subprocess.run(cmd, shell=True, check=check)


def run_script_if_exists(script_path, command):
    path = Path(script_path)
    if path.exists():
        run(command)
    else:
        print(f'SKIP: {script_path} is not implemented yet.')
        print(f'Planned command: {command}')


## Prediction Contract

Every downstream metric script expects NPZ files with `y_true` and `y_prob`, both shaped `(N, C)`. Store prediction files under `reports/revision/predictions/`.

In [ ]:
from pathlib import Path
import numpy as np

pred_dir = Path('reports/revision/predictions')
pred_dir.mkdir(parents=True, exist_ok=True)

for path in sorted(pred_dir.glob('*.npz')):
    data = np.load(path, allow_pickle=True)
    keys = sorted(data.files)
    print(path, keys)
    if {'y_true', 'y_prob'}.issubset(keys):
        print('  y_true:', data['y_true'].shape, 'y_prob:', data['y_prob'].shape)


## OOF And External Evaluation Commands

In [ ]:
RUN_HEAVY = False

commands = [
    ('scripts/eval_oof.py', 'python scripts/eval_oof.py'),
    ('scripts/eval_zeroshot.py', 'python scripts/eval_zeroshot.py'),
]

for script, command in commands:
    if RUN_HEAVY:
        run_script_if_exists(script, command)
    else:
        print(f'Heavy run disabled. Set RUN_HEAVY=True to execute: {command}')


## Re-run Inventory

In [ ]:
!python scripts/revision/05_artifact_inventory.py
